# Tech Challenge — Fase 4
## Etapa 4: Deploy — API RESTful (FastAPI) servindo o modelo LSTM

Cobre o **ponto 4**: uma API que recebe preços históricos de fechamento e devolve
a previsão do próximo preço, servindo o modelo no mesmo formato que a API de
produção usa — **ONNX**.

**Pré-requisito:** ter rodado a Etapa 2, que salvou `models/lstm_final.keras` e
`artifacts/ret_scaler.pkl` + `artifacts/inference_meta.pkl`, e depois
`python scripts/export_onnx.py`, que gera o `models/lstm_final.onnx`.

O `.keras` é o artefato de treino; o `.onnx` é derivado dele e é o que a API
carrega. Servir em ONNX dispensa o TensorFlow no runtime, o que reduz a imagem
de 1,64 GB para 495 MB e a memória residente de 722 MB para 159 MB — a diferença
entre caber ou não numa camada gratuita de hospedagem.

A API é **desenvolvida e testada aqui dentro** com `TestClient` (em processo, sem
subir servidor). A versão de produção vive em `app/`, com o mesmo contrato.

**Como o modelo prevê:** ele produz o *log-retorno* escalonado; a API reconstrói o
preço com `P̂ = P_último · exp(retorno)`. Por isso a entrada mínima é
`window_size + 1` preços (padrão 61) — para gerar os 60 retornos da janela.

## 0. Imports e configuração

In [1]:
import os
import numpy as np
import pandas as pd
import joblib

import onnxruntime as ort

from typing import List, Optional
from pydantic import BaseModel, Field
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

# Caminhos dos artefatos (ajuste se necessário)
MODEL_PATH  = "models/lstm_final.onnx"   # gerado por scripts/export_onnx.py
SCALER_PATH = "artifacts/ret_scaler.pkl"
META_PATH   = "artifacts/inference_meta.pkl"
CLEAN_CSV   = None  # definido após ler o meta (para o teste com dados reais)

/home/doglas/Documents/FIAP/fiap-tech-challenge-4/.venv/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 1. Inferência canônica (`Predictor`)

Toda a receita de inferência num só lugar — a mesma de `app/predictor.py`, para
API e notebook nunca divergirem: retorno → escala → modelo → inverse →
reconstrução. `horizon > 1` é recursivo (cada previsão vira input do dia seguinte).

A sessão ONNX é criada uma vez e reaproveitada. A entrada é convertida para
`float32`, que é o tipo do grafo exportado; os pesos já são `float32`, então não
há perda adicional.

In [2]:
class Predictor:
    def __init__(self, model_path=MODEL_PATH, scaler_path=SCALER_PATH, meta_path=META_PATH):
        self.session = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])
        self.input_name = self.session.get_inputs()[0].name
        self.scaler = joblib.load(scaler_path)
        self.meta   = joblib.load(meta_path)
        self.window = int(self.meta["window_size"])
        self.symbol = self.meta.get("symbol")

    @property
    def min_prices(self) -> int:
        return self.window + 1  # window+1 preços -> window retornos

    def predict(self, prices, horizon: int = 1):
        prices = np.asarray(prices, dtype=float)
        if prices.ndim != 1:
            raise ValueError("'prices' deve ser uma lista 1D de preços de fechamento.")
        if len(prices) < self.min_prices:
            raise ValueError(
                f"São necessários pelo menos {self.min_prices} preços "
                f"(janela={self.window}); recebidos {len(prices)}.")
        if np.any(prices <= 0):
            raise ValueError("Todos os preços devem ser positivos (o alvo é log-retorno).")

        recent = prices[-self.min_prices:]
        log_returns = np.diff(np.log(recent))
        window_scaled = self.scaler.transform(log_returns.reshape(-1, 1)).flatten()

        preds, last_price = [], float(recent[-1])
        for _ in range(int(horizon)):
            x = window_scaled.reshape(1, self.window, 1).astype(np.float32)
            r_scaled = float(self.session.run(None, {self.input_name: x})[0].reshape(-1)[0])
            r = float(self.scaler.inverse_transform([[r_scaled]])[0, 0])
            next_price = last_price * np.exp(r)
            preds.append(next_price)
            window_scaled = np.append(window_scaled[1:], r_scaled)  # rola a janela
            last_price = next_price
        return preds

## 2. Carregar o modelo

In [3]:
predictor = Predictor()
print(f"Modelo carregado | símbolo={predictor.symbol} | janela={predictor.window} "
      f"| mínimo de preços={predictor.min_prices}")

Modelo carregado | símbolo=AAPL | janela=60 | mínimo de preços=61


## 3. Teste da inferência com dados reais

Pegamos os últimos preços do CSV limpo da Etapa 1 para um teste realista.

In [4]:
CLEAN_CSV = os.path.join("artifacts", f"{predictor.symbol}_clean.csv")
df = pd.read_csv(CLEAN_CSV, index_col=0, parse_dates=True)
ultimos = df["Close"].astype(float).values[-(predictor.min_prices):]

previsao = predictor.predict(ultimos, horizon=1)[0]
print(f"Último preço real:   {ultimos[-1]:.2f}")
print(f"Preço previsto (D+1): {previsao:.2f}")
print("Previsão 5 dias:", [round(float(x), 2) for x in predictor.predict(ultimos, horizon=5)])

Último preço real:   298.01
Preço previsto (D+1): 297.71
Previsão 5 dias: [297.71, 297.74, 298.1, 298.58, 299.04]


## 4. API (FastAPI)

Contrato de entrada/saída validado com Pydantic. Endpoints:
- `GET /health` — status e contrato (nº mínimo de preços).
- `POST /predict` — recebe `prices` (mais antigo → mais recente) e `horizon`.

In [5]:
class PredictRequest(BaseModel):
    prices: List[float] = Field(..., description="Fechamentos, do mais antigo ao mais recente.",
                                examples=[[150.0, 151.2, 149.8]])
    horizon: int = Field(1, ge=1, le=30, description="Dias à frente (recursivo).")
    symbol: Optional[str] = Field(None, description="Ação a que os preços se referem; apenas rotula a resposta.")

class PredictResponse(BaseModel):
    # 'symbol' é a ação que a requisição diz representar;
    # 'model_trained_on' é a ação em que o modelo foi treinado.
    symbol: Optional[str] = None
    model_trained_on: Optional[str] = None
    last_price: float
    horizon: int
    predictions: List[float]

app = FastAPI(title="Tech Challenge Fase 4 — API de Previsão (LSTM)", version="1.0.0")

@app.get("/health")
def health():
    return {"status": "ok", "model_trained_on": predictor.symbol,
            "window_size": predictor.window, "min_prices": predictor.min_prices}

@app.post("/predict", response_model=PredictResponse)
def predict(req: PredictRequest):
    try:
        preds = predictor.predict(req.prices, horizon=req.horizon)
    except ValueError as e:
        raise HTTPException(status_code=422, detail=str(e))
    return PredictResponse(symbol=req.symbol, model_trained_on=predictor.symbol,
                           last_price=float(req.prices[-1]),
                           horizon=req.horizon, predictions=[float(x) for x in preds])

## 5. Testar a API em processo (`TestClient`)

`TestClient` executa a app sem subir servidor — ótimo para validar dentro do
notebook. Testamos o caminho feliz e as validações de erro.

In [6]:
client = TestClient(app)

print("GET /health ->", client.get("/health").json())

r1 = client.post("/predict", json={"prices": list(ultimos), "horizon": 1})
print("POST /predict h1 ->", r1.status_code, r1.json())

r5 = client.post("/predict", json={"prices": list(ultimos), "horizon": 5})
print("POST /predict h5 ->", r5.status_code, "| preds:",
      [round(x, 2) for x in r5.json()["predictions"]])

# validações
poucos = client.post("/predict", json={"prices": list(ultimos[:10]), "horizon": 1})
print("poucos preços ->", poucos.status_code, poucos.json()["detail"])

neg = client.post("/predict", json={"prices": [-1.0] + list(ultimos), "horizon": 1})
print("preço negativo ->", neg.status_code, neg.json()["detail"])

badh = client.post("/predict", json={"prices": list(ultimos), "horizon": 999})
print("horizon inválido ->", badh.status_code)

GET /health -> {'status': 'ok', 'model_trained_on': 'AAPL', 'window_size': 60, 'min_prices': 61}
POST /predict h1 -> 200 {'symbol': None, 'model_trained_on': 'AAPL', 'last_price': 298.010009765625, 'horizon': 1, 'predictions': [297.70946715163353]}
POST /predict h5 -> 200 | preds: [297.71, 297.74, 298.1, 298.58, 299.04]
poucos preços -> 422 São necessários pelo menos 61 preços (janela=60); recebidos 10.
preço negativo -> 422 Todos os preços devem ser positivos (o alvo é log-retorno).
horizon inválido -> 422
